In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# KITTI Object Detection - Training in Colab\n",
    "\n",
    "ноут автоматически клонирует репозиторий, устанавливает зависимости и запускает обучение моделей.\n",
    "\n",
    "**Автор:** [chase]\n",
    "\n",
    "**Дата:** 2026"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Подключение Google Drive\n",
    "\n",
    "Подключаем Google Drive, чтобы сохранять результаты обучения."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Подключаем Google Drive\n",
    "from google.colab import drive\n",
    "drive.mount('/content/drive')\n",
    "print(\"Google Drive подключен!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Клонирование репозитория\n",
    "\n",
    "Клонируем ваш репозиторий с GitHub в среду Colab."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Клонируем репозиторий (ЗАМЕНИТЕ ССЫЛКУ НА ВАШУ!)\n",
    "!git clone https://github.com/mylisadfran13/kitti-object-detection.git\n",
    "%cd kitti-object-detection\n",
    "print(\"Репозиторий клонирован!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Установка зависимостей\n",
    "\n",
    "Устанавливаем все необходимые библиотеки."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Устанавливаем зависимости\n",
    "!pip install -q torch torchvision ultralytics opencv-python matplotlib seaborn pandas numpy tqdm pyyaml tensorboard torchmetrics scikit-learn albumentations\n",
    "print(\"Библиотеки установлены!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Проверка GPU\n",
    "\n",
    "Проверяем, доступен ли GPU для ускорения обучения."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Проверяем GPU\n",
    "import torch\n",
    "print(f\"CUDA доступен: {torch.cuda.is_available()}\")\n",
    "if torch.cuda.is_available():\n",
    "    print(f\"Модель GPU: {torch.cuda.get_device_name(0)}\")\n",
    "    print(f\"Количество GPU: {torch.cuda.device_count()}\")\n",
    "else:\n",
    "    print(\"GPU не найден. Будет использован CPU (медленнее).\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Создание папки для результатов\n",
    "\n",
    "Создаем папку на Google Drive для сохранения обученных моделей и графиков."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Создаем папку для результатов\n",
    "import os\n",
    "save_path = '/content/drive/MyDrive/kitti_results/'\n",
    "os.makedirs(save_path, exist_ok=True)\n",
    "print(f\"Результаты будут сохранены в: {save_path}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Обучение YOLOv8\n",
    "\n",
    "Запускаем обучение модели YOLOv8n на датасете KITTI.\n",
    "\n",
    "**Параметры:**\n",
    "- Модель: YOLOv8n (nano) — самая маленькая и быстрая\n",
    "- Эпохи: 10 (для быстрого теста)\n",
    "- Размер изображения: 640×640\n",
    "- Batch size: 8 (маленький, чтобы не упасть по памяти)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Обучение YOLOv8\n",
    "from ultralytics import YOLO\n",
    "import torch\n",
    "\n",
    "print(\"Загрузка модели YOLOv8n...\")\n",
    "model = YOLO('yolov8n.pt')\n",
    "print(\"Модель YOLOv8n загружена!\")\n",
    "\n",
    "print(\"Обучение начинается...\")\n",
    "results = model.train(\n",
    "    data='configs/kitti.yaml',\n",
    "    epochs=10,\n",
    "    imgsz=640,\n",
    "    batch=8,\n",
    "    device=0 if torch.cuda.is_available() else 'cpu',\n",
    "    project='results/yolo',\n",
    "    name='exp'\n",
    ")\n",
    "print(\"Обучение завершено!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Сохранение модели\n",
    "\n",
    "копируется лучшая модель на Google Drive."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Сохраняем модель на Google Drive\n",
    "!cp results/yolo/exp/weights/best.pt {save_path}yolo_best.pt\n",
    "print(f\"Модель сохранена в: {save_path}yolo_best.pt\")\n",
    "\n",
    "# Проверяем размер файла\n",
    "import os\n",
    "size = os.path.getsize(f'{save_path}yolo_best.pt') / (1024 * 1024)\n",
    "print(f\"Размер модели: {size:.2f} МБ\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. Визуализация предсказаний\n",
    "\n",
    "Тестируем модель на одном изображении из KITTI и сохраняем результат."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Визуализация предсказаний\n",
    "import cv2\n",
    "import matplotlib.pyplot as plt\n",
    "import os\n",
    "\n",
    "img_path = 'data/raw/training/image_2/000000.png'\n",
    "\n",
    "if os.path.exists(img_path):\n",
    "    # Делаем предсказание\n",
    "    results = model(img_path)\n",
    "    annotated = results[0].plot()\n",
    "    \n",
    "    # Сохраняем\n",
    "    plt.figure(figsize=(12, 8))\n",
    "    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))\n",
    "    plt.title('YOLOv8 Detection on KITTI', fontsize=14)\n",
    "    plt.axis('off')\n",
    "    \n",
    "    # Сохраняем локально и на Google Drive\n",
    "    os.makedirs('results/plots', exist_ok=True)\n",
    "    plt.savefig('results/plots/prediction.png', dpi=150, bbox_inches='tight')\n",
    "    plt.savefig(f'{save_path}prediction.png', dpi=150, bbox_inches='tight')\n",
    "    print(f\"Визуализация сохранена в: {save_path}prediction.png\")\n",
    "    \n",
    "    # Показываем в ноутбуке\n",
    "    plt.show()\n",
    "else:\n",
    "    print(f\"Изображение не найдено: {img_path}\")\n",
    "    print(\"Загрузите датасет KITTI в папку data/raw/\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 9. Оценка модели\n",
    "\n",
    "Вычисляем метрики качества на валидационной выборке."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Оценка модели\n",
    "print(\"Оценка модели на валидационной выборке...\")\n",
    "metrics = model.val()\n",
    "\n",
    "# Выводим результаты\n",
    "print(\"=\" * 50)\n",
    "print(\"РЕЗУЛЬТАТЫ ОЦЕНКИ\")\n",
    "print(\"=\" * 50)\n",
    "print(f\"mAP@0.5: {metrics.box.map50:.4f}\")\n",
    "print(f\"mAP@0.5:0.95: {metrics.box.map:.4f}\")\n",
    "print(f\"Precision: {metrics.box.mp:.4f}\")\n",
    "print(f\"Recall: {metrics.box.mr:.4f}\")\n",
    "print(\"=\" * 50)\n",
    "\n",
    "# Сохраняем метрики в файл\n",
    "with open(f'{save_path}metrics.txt', 'w') as f:\n",
    "    f.write(f\"mAP@0.5: {metrics.box.map50:.4f}\\n\")\n",
    "    f.write(f\"mAP@0.5:0.95: {metrics.box.map:.4f}\\n\")\n",
    "    f.write(f\"Precision: {metrics.box.mp:.4f}\\n\")\n",
    "    f.write(f\"Recall: {metrics.box.mr:.4f}\\n\")\n",
    "print(f\"Метрики сохранены в: {save_path}metrics.txt\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 10. Готово!\n",
    "\n",
    "Обучение завершено! Все результаты сохранены на Google Drive.\n",
    "\n",
    "**Что сохранено:**\n",
    "- Модель: `kitti_results/yolo_best.pt`\n",
    "- Визуализация: `kitti_results/prediction.png`\n",
    "- Метрики: `kitti_results/metrics.txt`"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"=\" * 50)\n",
    "print(\"ВСЕ ГОТОВО!\")\n",
    "print(\"=\" * 50)\n",
    "print(f\"Результаты на Google Drive: {save_path}\")\n",
    "print(\"Модель: yolo_best.pt\")\n",
    "print(\"Визуализация: prediction.png\")\n",
    "print(\"Метрики: metrics.txt\")\n",
    "print(\"=\" * 50)"
   ]
  }
 ],
 "metadata": {
  "colab": {
   "provenance": []
  },
  "kernelspec": {
   "display_name": "Python 3",
   "name": "python3"
  },
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 0
}